# ARCSIX production example

Install the repository into this notebook's Python environment first:
`python -m pip install -e /path/to/SizeDistMerge`.
The Python files live directly in `src/`; the public import remains `sizedistmerge`.
No private data or absolute machine-specific paths are included.

This example shows how to call the shared ARCSIX processing helpers for
one-minute periods. It is not the archived R1 or R2 production notebook and does
not promise to reproduce either released product. Choose campaign inputs,
sampling criteria, fitting settings, weights and QC thresholds deliberately.
All file-writing steps are disabled by default. No fallback dataset is silently
substituted for missing FIMS observations.

In [ ]:
# Calibration for these data; edit when using a different calibration.
RI_UHSAS_SRC = complex(1.52, 0.0)
RI_POPS_SRC = complex(1.615, 0.001)
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from sizedistmerge import optical_diameter as od

# Locate the checkout from its root or notebooks/ directory.
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'pyproject.toml').is_file() and (p / 'campaign_merge_production').is_dir())
sys.path.insert(0, str(REPO))
from campaign_merge_production import arcsix_merge_production as mp

DATA_DIR = None  # Set to Path('/your/ARCSIX_P3B') before running.
OUTPUT_DIR = REPO / 'local' / 'arcsix_example_output'
RUN_MERGE = False
RUN_QC = False

boundaries = pd.date_range('2024-06-05 11:30:00', periods=6, freq='1min')
periods = list(zip(boundaries[:-1], boundaries[1:]))

In [ ]:
if RUN_MERGE:
    if DATA_DIR is None or not Path(DATA_DIR).is_dir():
        raise ValueError('Set DATA_DIR to the campaign input directory.')
    if OUTPUT_DIR.exists():
        raise FileExistsError('Choose a new output directory for this example.')
    mp.run_arcsix_merge_for_periods(
        periods, data_dir=DATA_DIR, output_dir=OUTPUT_DIR,
        include_pops=True, pops_ri_src=RI_POPS_SRC,
        lut_dir=REPO / 'lut', moment='V', space='linear',
        bounds_uhsas=((1.3, 1.8),), bounds_aps=((950., 2000.),),
        fims_lag=10, min_samples_per_inst=10,
        temporal_w_uh=0., temporal_w_po=0., temporal_w_rho=0.,
        output_edges=np.geomspace(10., 5000., 101),
        save_time_series_plots=False, save_loss_plots=False,
        save_merge_plots=False, resume=False,
        uhsas_ri_src=RI_UHSAS_SRC,
    )
else:
    print('Merge disabled. Configure inputs and review settings before running.')

## Quality control
QC needs enough periods to estimate concentration-difference thresholds.
The five-minute example is too short for a representative QC assessment;
use a suitably longer run. Review the returned warnings before publishing data.
ICARTT export is a separate operation through
`mp.convert_qc_netcdf_to_icartt`; assign publication metadata and a revision
only after QC and scientific review.

In [ ]:
if RUN_QC:
    if DATA_DIR is None:
        raise ValueError('Set DATA_DIR before QC.')
    qc_result = mp.run_post_merge_product_qc(
        OUTPUT_DIR, DATA_DIR, min_points_for_robust=50)
    print(qc_result)